# Computation A — the CKM-extended charged current

**It from Bit via Gödel, Paper 1** · companion to `sec:interactions` and `app:qiskit-ckm` · addresses **op:cabibbo**

Beta decay on the active quark, extended with a generation qubit `g`. The $W^-$ gate is the $T_3$
raise on $B_q$ tensored with the CKM rotation $R_y(2\delta)$ on `g`, with $\delta = 2/9$ — the
Koide phase. The claim under test: the circuit's output amplitudes *are* the CKM factors,
$|V_{ud}| = \cos(2/9)$ and $|V_{us}| = \sin(2/9)$.

Wire order (little-endian): `q0=B_nu, q1=B_e, q2=g, q3=C_q, q4=B_q, q5=A_q`.

**Epistemic status.** Execution validates the *encoding* of the framework's claim, not the claim —
that is the paper's burden. The exact-statevector cell reproduces the verified appendix numbers;
the sampled run (local or hardware) estimates the same quantities from counts.

In [1]:
USE_HARDWARE = False   # flip to True to run on IBM hardware
SHOTS = 8192

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

DELTA = 2/9  # the Koide phase, radians

qc = QuantumCircuit(6, name="beta_decay_CKM")
qc.ry(2*DELTA, 2)          # CKM rotation on the generation register
qc.x(4)                    # T3 raise: d -> up-type
qc.h(1); qc.cx(1, 0)       # W- decays to the lepton Bell pair

qc_meas = qc.copy()
qc_meas.measure_all()
qc.draw()

┌───┐
q_0: ───────────────┤ X ├
          ┌───┐     └─┬─┘
q_1: ─────┤ H ├───────■──
     ┌────┴───┴────┐     
q_2: ┤ Ry(0.44444) ├─────
     └─────────────┘     
q_3: ────────────────────
          ┌───┐          
q_4: ─────┤ X ├──────────
          └───┘          
q_5: ────────────────────

## Exact statevector (the verified baseline)

Four basis states, two generation branches spread over the two Bell components. Extracted:
$|V_{ud}| = 0.97541$ (+0.11% vs PDG 0.97435), $|V_{us}| = 0.22040$ (−1.74% vs 0.22430).

In [2]:
psi = Statevector(qc)
for i, a in enumerate(psi.data):
    if abs(a) > 1e-12:
        print(f"|{i:06b}>  amplitude {a.real:+.6f}")
amps = psi.data
print(f"\nexact |V_ud| = {np.sqrt(2)*abs(amps[0b010000]):.5f}"
      f"   |V_us| = {np.sqrt(2)*abs(amps[0b010100]):.5f}")

|010000>  amplitude +0.689719
|010011>  amplitude +0.689719
|010100>  amplitude +0.155845
|010111>  amplitude +0.155845

exact |V_ud| = 0.97541   |V_us| = 0.22040


## Sampled run — local simulator or IBM hardware

On hardware the statevector is not available; the CKM factors are read from the **generation-qubit
marginal**: $|V_{ud}| = \sqrt{P(q_2{=}0)}$, $|V_{us}| = \sqrt{P(q_2{=}1)}$. The Bell-pair quality
check $P(q_1{=}q_0)$ comes from the same counts.

Hardware notes: the circuit is 6 qubits, depth ~4 — easy for any current device. Expect readout
error to bias $P(q_2{=}1) \approx 0.049$ upward by the order of the per-qubit readout error
(1–2% absolute), inflating $|V_{us}|$; measurement-error mitigation is the natural first
refinement.

In [3]:
if USE_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    service = QiskitRuntimeService()
    backend = service.least_busy(simulator=False, operational=True)
    print("backend:", backend.name)
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    isa = pm.run(qc_meas)
    counts = Sampler(mode=backend).run([isa], shots=SHOTS).result()[0].data.meas.get_counts()
else:
    from qiskit.primitives import StatevectorSampler
    counts = StatevectorSampler().run([qc_meas], shots=SHOTS).result()[0].data.meas.get_counts()

total = sum(counts.values())
print(f"{len(counts)} distinct outcomes from {total} shots")

4 distinct outcomes from 8192 shots


In [4]:
# bitstrings are little-endian: b[-1-i] is qubit i
P_g1   = sum(c for b, c in counts.items() if b[-3] == '1') / total
V_ud_s = np.sqrt(1 - P_g1); V_us_s = np.sqrt(P_g1)
bell   = sum(c for b, c in counts.items() if b[-1] == b[-2]) / total

print(f"sampled |V_ud| = {V_ud_s:.5f}   (prediction cos(2/9) = {np.cos(DELTA):.5f}, PDG 0.97435)")
print(f"sampled |V_us| = {V_us_s:.5f}   (prediction sin(2/9) = {np.sin(DELTA):.5f}, PDG 0.22430)")
print(f"Bell-pair check P(q1==q0) = {bell:.4f}   (ideal 1.0)")

sampled |V_ud| = 0.97566   (prediction cos(2/9) = 0.97541, PDG 0.97435)
sampled |V_us| = 0.21931   (prediction sin(2/9) = 0.22040, PDG 0.22430)
Bell-pair check P(q1==q0) = 1.0000   (ideal 1.0)


## What to vary

The comparison row is the test the framework must pass or fail: replace `DELTA` and watch
$|V_{ud}|$ move. The Cabibbo-suppressed branch ($q_2{=}1$) is present with the correct relative
weight even though real beta decay cannot reach charm — kinematics lives in the spacetime sector,
and the circuit computes the internal-structure factor only. Extending `g` to two qubits and the
rotation to a $3\times3$ unitary built from the framework's phases is the full-CKM version of the
same handle.